### Day 17 · Pandas 进阶(分组聚合 / 合并 / 塑形 / 时间序列 / 文本 / IO)**前置**:Day16 已掌握 Series / DataFrame / 索引 / 过滤 / 排序。**目标**:从"问一列"升级到"问整张大表"——分组聚合、表合并、数据重塑、时间序列、文本处理、分类数据、数据导入导出,覆盖 Pandas 日常 80% 的工作。**学习顺序**:①分组聚合 ②合并 ③concat ④透视表 ⑤重塑 ⑥时间序列→ ⑦文本处理 → ⑧分类数据 → ⑨导入导出。每个知识点独立,按序看即可。

#### 分组聚合 groupby + agg "按类别汇总"**痛点**:查"全班平均成绩"容易,但想"每个班同时看平均分、最高分、人数",难道写 3 个 query 再手工拼起来?**类比**:班级大合影,老师喊"按小组站好"(split),每组统计人数(apply),再汇总一张清单(combine)。这就是 **拆分 → 应用 → 合并**三步曲。**解释**:`groupby("列名")` 把行按该列的唯一值切成若干子集,返回 GroupBy 对象(此时还没算),再调用 `agg()` 一次性传进多个聚合函数。函数列表自动生成列名,命名聚合用 `新列=("原列", "函数名")` 自定义列名。> **ASCII 内存图**>> ```> df.groupby("班级"):>   ┌─ 原 DataFrame (5 行) ───────────────────┐>   │ 班级=[A,A,B,B,B]  成绩=[88,92,75,85,90] │>   └───────────────────┬──────────────────────┘>                       │ 按"班级"切分>        ┌──────────────┴──────────────┐>        ▼                            ▼>   ┌─────────┐                 ┌─────────┐>   │ A 组    │                 │ B 组    │>   │ 88, 92  │                 │ 75,85,90│>   └────┬────┘                 └────┬────┘>        │ .mean()                  │ .mean()>        ▼                          ▼>   Series: 索引=班级>   A: 90.0   B: 83.33> ```

In [ ]:
import pandas as pd

# 创建成绩表:5 名学生,分属两个班级
df = pd.DataFrame({
    "班级": ["A", "A", "B", "B", "B"],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "成绩": [88, 92, 75, 85, 90]
})

# 按班级分组,取成绩列,求均值
result = df.groupby("班级")["成绩"].mean()
print(result)

# --- 执行过程 ---
# 第 1 行 df.groupby("班级"):
#   ① 找"班级"列的唯一值 → A、B
#   ② 按值把行拆成两组(不计算)
#   ③ 返回 GroupBy 对象
#
# 第 2 行 ["成绩"]:
#   ① 在 GroupBy 对象上再选一列
#   ② 接下来的聚合只对这一列生效
#
# 第 3 行 .mean():
#   ① 对 A 组 (88,92) 求均值 → 90.0
#   ② 对 B 组 (75,85,90) 求均值 → 83.33
#   ③ 返回 Series,索引 = 分组键(班级)


**逐行解剖**- `groupby("班级")`:惰性计算,直到调用聚合才真正算。类似 SQL 的 `GROUP BY`。- `["成绩"]`:在聚合前先锁定目标列;不写则对所有数值列都聚合。- `.mean()`:常用聚合函数还有 `.sum() .max() .min() .count() .std()` 等。- 返回 Series;如果希望输出 DataFrame,去掉 `["成绩"]` 或改用 `agg`。

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "班级": ["A", "A", "B", "B", "B"],
    "成绩": [88, 92, 75, 85, 90]
})

# 写法 1:agg 传函数列表,自动生成列名
res1 = df.groupby("班级")["成绩"].agg(["mean", "max", "min"])
print("写法 1 — 列表(自动列名):")
print(res1)
print("---")

# 写法 2:命名聚合,自定义输出列名
res2 = df.groupby("班级").agg(
    平均分=("成绩", "mean"),
    最高分=("成绩", "max"),
    人数=("成绩", "count")
)
print("写法 2 — 命名聚合:")
print(res2)

# --- 执行过程 ---
# 写法 1 - agg(["mean","max","min"]):
#   ① 列表里每个函数独立作用于同一列
#   ② 列名直接取函数名 → mean/max/min
#
# 写法 2 - 命名聚合:
#   ① 等号左边是新列名,右边是被聚合列+函数
#   ② 可以同时写多个,比列表清晰得多
#   ③ 结果列名是 平均分/最高分/人数(自定义)


**逐行解剖(Agg)**- `.agg(["mean","max","min"])`:列表形式,适合快速看分布,列名是函数名。- `.agg(平均分=("成绩","mean"))`:命名聚合(Pandas 0.25+),左新列右(value,func)。- 不选列时,`groupby(...).agg(...)` 会**对所有数值列**同时聚合。- `count` 不计 NaN;`size` 计入 NaN,缺数据时两者不一致。

> **常见错误**> 1. **错误现象**:对 GroupBy 对象直接 print,输出 `<pandas.core.groupby...>`>    **原因**:groupby 是惰性的,必须调用聚合才真正计算。> 2. **错误现象**:命名聚合报错 `ValueError: cannot aggregate...`>    **原因**:元组里原列名写错,确认原列确实存在且拼写一致。> 3. **错误现象**:聚合后列名变成了函数名,不是期望的中文名>    **原因**:用了列表 agg,改用命名聚合即可自定义。

**练习**超市销售表已有"品类"和"销售额"两列,请用 groupby + 命名聚合输出每品类:`总销售额=sum`、`平均销售额=mean`、`订单数=count`。> 问自己:> - 命名聚合里原列应该是哪一列?> - 如果想统计订单笔数,应该用 count 还是 size?> - 如果对多列(城市和品类)同时分组,groupby 参数怎么写?

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "品类": ["水果", "水果", "零食", "零食", "饮料"],
    "销售额": [120, 150, 80, 60, 95]
})

# TODO:用命名聚合输出总销售额、平均销售额、订单数
pass


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "品类": ["水果", "水果", "零食", "零食", "饮料"],
    "销售额": [120, 150, 80, 60, 95]
})

result = df.groupby("品类").agg(
    总销售额=("销售额", "sum"),
    平均销售额=("销售额", "mean"),
    订单数=("销售额", "count")
)
print(result)


#### merge —— 像 SQL 一样"按 key"把两张表拼起来**痛点**:学生信息在一张表,成绩在另一张表,如何像 Excel VLOOKUP 一样拼起来?**类比**:两张成绩单通过"学号"这条线左右对齐。- `inner`:只留都有的学号- `left`:以左表为主,右表匹配不到填 NaN- `right`:以右表为主,反过来- `outer`:全都要,匹配不到填 NaN**解释**:`merge(left, right, on="键列", how="连接类型")`。- `on`:两张表都有的共同列。键名不同用 `left_on` / `right_on`。- `how`:inner / left / right / outer。默认 inner。> **ASCII 内存图**>> ```> 左表 df_stu:             右表 df_score:> S1 张三                  S1 88> S2 李四                  S2 92> S3 王五                  S3 75> S4 赵六                  S5 80>> pd.merge(left=df_stu, right=df_score, on="学号", how="inner"):>   学号  姓名  成绩>   S1   张三   88>   S2   李四   92>   S3   王五   75> (S4 和 S5 被丢弃)> ```

In [ ]:
import pandas as pd

# 左表:学生基本信息,学号 S1-S4
df_stu = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S4"],
    "姓名": ["张三", "李四", "王五", "赵六"]
})

# 右表:学生成绩(S4 没成绩,S5 没信息)
df_score = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S5"],
    "成绩": [88, 92, 75, 80]
})

# inner:只保留两边都有的学号
inner = pd.merge(df_stu, df_score, on="学号", how="inner")
print("inner(默认):", inner.shape)
print(inner)

# --- 执行过程 ---
# pd.merge(df_stu, df_score, on="学号", how="inner"):
#   ① 以"学号"列为键,左右对齐
#   ② inner 取交集:S1/S2/S3 都在两边
#   ③ S4(只在左)和 S5(只在右)被丢弃
#   ④ 每行:学号+左表姓名+右表成绩


**逐行解剖**- `on="学号"`:连接键,两张表都有的列名。键名不同用 `left_on="l" / right_on="r"`。- `how="inner"`:最常用,但可能丢数据(缺失的行会被删)。- `how="left"`:左表为主不丢;右表匹配不上就填 NaN。- 一对多匹配会**行数膨胀**,每次 merge 后立即检查 `.shape`。

In [ ]:
import pandas as pd

df_stu = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S4"],
    "姓名": ["张三", "李四", "王五", "赵六"]
})

df_score = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S5"],
    "成绩": [88, 92, 75, 80]
})

# left:以左表为主,匹配不上填 NaN
left = pd.merge(df_stu, df_score, on="学号", how="left")
print("left: S4 成绩是 NaN")
print(left)
print("---")

# outer:取并集,全部保留,匹配不上填 NaN
outer = pd.merge(df_stu, df_score, on="学号", how="outer")
print("outer: S4 成绩NaN, S5 姓名NaN")
print(outer)

# --- 执行过程 ---
# left join 过程:
#   ① 左表 S1~S4 全部保留
#   ② S4 找不到成绩 → 填 NaN
#   ③ 右表中 S5 被丢弃(不关心右表)
#
# outer join 过程:
#   ① 保留两边全部学号(S1~S5)
#   ② 左侧独有的右列 NaN;右侧独有的左列 NaN


**逐行解剖(进阶)**- **left join**:保留左表所有行,适合"基于主表扩展字段"。- **outer join**:取并集,一行不丢,但可能大量 NaN。- **validate 参数**:`validate="one_to_one"` 发现键重复立刻报错,  防止一对多意外膨胀。- `indicator=True`:加一列 `_merge` 标明每行来自哪侧(left_only/both)。

> **常见错误**> 1. **错误现象**:merge 后行数变多(一对多膨胀)>    **原因**:键列有重复值。用 `validate="one_to_one"` 排查。> 2. **错误现象**:合并后出现 `列名_x` / `列名_y` 后缀>    **原因**:两张表除 on 列外还有别的同名列,自动加后缀区分。> 3. **错误现象**:`ValueError: On column must be unique`>    **原因**:on 指定的列不存在,或 left/right 写法颠倒。

**练习**员工表有"员工号/姓名",部门表有"员工号/部门"。请用 left join 合并,行数是多少?哪些员工没有部门(用 `.isna()` 判断)?> 问自己:> - left join 的主表应该是员工表还是部门表?> - 合并后 NaN 在"部门"列还是"姓名"列?> - 如何快速统计没匹配上的行数?

In [ ]:
import pandas as pd

df_emp = pd.DataFrame({"员工号": ["E1", "E2", "E3", "E4"], "姓名": ["张三", "李四", "王五", "赵六"]})

df_dept = pd.DataFrame({"员工号": ["E1", "E3", "E5"], "部门": ["技术", "市场", "财务"]})

# TODO:用 left join 合并,检查 shape,找没匹配上的行
pass


In [ ]:
import pandas as pd

df_emp = pd.DataFrame({"员工号": ["E1", "E2", "E3", "E4"], "姓名": ["张三", "李四", "王五", "赵六"]})

df_dept = pd.DataFrame({"员工号": ["E1", "E3", "E5"], "部门": ["技术", "市场", "财务"]})

merged = pd.merge(df_emp, df_dept, on="员工号", how="left")
print("合并后 shape:", merged.shape)
print(merged)

# 找没匹配上的员工(部门列为 NaN)
missing = merged[merged["部门"].isna()]
print("没匹配上的员工:")
print(missing)


#### concat 沿轴拼多个 DataFrame(粘纸)**痛点**:12 张月报表(列都一样)纵向堆成年度大表;或两张表列不同,横向拼在一起。**类比**:把两张纸粘起来——- `axis=0`:上下粘(加**行**,最常见的"合并月报表")- `axis=1`:左右粘(加**列**,像 Excel 旁边插入一列)**解释**:`pd.concat([df1, df2], axis=0, ignore_index=True)`。- `axis=0`:纵向拼(默认),列对齐、行数相加。- `axis=1`:横向拼,索引对齐,列数相加。- `ignore_index=True`:抛弃原索引,重新生成连续整数索引。与 merge 区别:- concat 只拼"形状",不关心键。想"粘数据"用 concat。- merge 按 key 对齐(像 SQL JOIN)。想"查关系"用 merge。

In [ ]:
import pandas as pd

# 两个结构完全相同的 DataFrame
df1 = pd.DataFrame({"A": [1, 2], "B": [3, 4]})
df2 = pd.DataFrame({"A": [5, 6], "B": [7, 8]})

# 纵向堆叠(axis=0),ignore_index 重置索引
stacked = pd.concat([df1, df2], axis=0, ignore_index=True)
print("纵向堆叠(4 行):")
print(stacked)

# --- 执行过程 ---
# pd.concat([df1,df2], axis=0, ignore_index=True):
#   ① 把 df2 的行接到 df1 后面
#   ② axis=0 → 纵向(行数叠加)
#   ③ ignore_index=True → 索引变成 0,1,2,3 连续
#   如果没有 ignore_index:索引仍是 0,1,0,1(重复)


**逐行解剖**- `axis=0`(默认):**上下堆叠**,最常用。行数 = 各 DataFrame 行数之和。- `ignore_index=True`:**强烈推荐**,不然原索引保留、可能出现重复。- 列不一致时,多出来的列在另一方填 NaN(纵向)或新列(横向)。

In [ ]:
import pandas as pd

# 横向拼(axis=1):列不同,按索引对齐
df_left = pd.DataFrame({"A": [1, 2], "B": [3, 4]})
df_right = pd.DataFrame({"C": [5, 6], "D": [7, 8]})

# axis=1:按左右位置合并,索引自动对齐
wide = pd.concat([df_left, df_right], axis=1)
print("横向拼接(4 列):")
print(wide)

# --- 执行过程 ---
# pd.concat([df_left, df_right], axis=1):
#   ① 把 df_right 的列放在 df_left 的右边
#   ② 两边都是索引 0,1 → 自动对齐
#   ③ 结果得到 A,B,C,D 四列
#
# 如果两边行数不同,多出的行会用 NaN 补齐


> **常见错误**> 1. **错误现象**:concat 后索引重复(如 0,1,0,1),`.loc[0]` 返回多行>    **原因**:忘写 `ignore_index=True`。加上即可重置为连续整数。> 2. **错误现象**:横向拼后出现一片 NaN,数据没有对齐>    **原因**:两张表索引不一致(一份从 1 开始、一份从 0 开始)。  解决:先调用 `reset_index(drop=True)` 统一索引。

**练习**给定 3 个班的成绩小表格(列都是"姓名/成绩"),用 concat 纵向合并成年级大表,索引重新编号。> 问自己:> - axis 应该传 0 还是 1?> - ignore_index 不设会怎样?> - 如果某个班级多了"性别"列,还能直接纵向堆叠吗?

In [ ]:
import pandas as pd

df_a = pd.DataFrame({"姓名": ["张三", "李四"], "成绩": [88, 92]})
df_b = pd.DataFrame({"姓名": ["王五"], "成绩": [75]})
df_c = pd.DataFrame({"姓名": ["赵六", "孙七"], "成绩": [85, 90]})

# TODO:concat 三个班级,ignore_index 重置索引
pass


In [ ]:
import pandas as pd

df_a = pd.DataFrame({"姓名": ["张三", "李四"], "成绩": [88, 92]})
df_b = pd.DataFrame({"姓名": ["王五"], "成绩": [75]})
df_c = pd.DataFrame({"姓名": ["赵六", "孙七"], "成绩": [85, 90]})

all_df = pd.concat([df_a, df_b, df_c], axis=0, ignore_index=True)
print("年级大表,shape:", all_df.shape)
print(all_df)


#### pivot_table 把长表盘成"行×列"的矩阵**痛点**:销售表每行是一条记录,想看"城市×季度"的交叉汇总表,怎么写?**类比**:长条数据像一根挂面(只有日期/城市/销量三列),透视表把它盘成一张网格(行=日期、列=城市、值=销量)。**解释**:`df.pivot_table(index=, columns=, values=, aggfunc=)`。- `index`:哪个列变成行的索引。- `columns`:哪个列变成列名(展开)。- `values`:哪个列的数值填进格子。- `aggfunc`:同一格子多行时怎么聚合(sum/mean/count)。> **ASCII 内存图**>> ```> 原长表(4 行):>  日期    城市    销量>  周一    北京    120>  周一    上海     85>  周二    北京     96>  周二    上海    110>> pivot_table(index="日期", columns="城市", values="销量", aggfunc="sum"):>              城市=北京   城市=上海>  日期=周一     120          85>  日期=周二      96         110>  (矩阵:行=日期, 列=城市, 格子=销量)> ```

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "日期": ["周一", "周一", "周二", "周二"],
    "城市": ["北京", "上海", "北京", "上海"],
    "销量": [120, 85, 96, 110]
})

pt = df.pivot_table(
    index="日期",
    columns="城市",
    values="销量",
    aggfunc="sum"
)
print("透视表:")
print(pt)

# --- 执行过程 ---
# pivot_table(index=日期, columns=城市, values=销量, aggfunc=sum):
#   ① "日期"唯一值变成行 → 周一、周二
#   ② "城市"唯一值变成列 → 北京、上海
#   ③ 每个(日期,城市)格子填入对应销量
#   ④ 若多条记录落同一格子,用 sum 聚合


**逐行解剖**- `index` + `columns` 共同决定一个网格单元。某组合出现多次,  就用 `aggfunc` 汇总。- 缺组合默认填 NaN,可通过 `fill_value=0` 改成 0。- `margins=True`:给透视表加上行列汇总(类似 Excel 的"合计"行)。

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "日期": ["周一", "周一", "周一", "周二", "周二"],
    "城市": ["北京", "北京", "上海", "北京", "上海"],
    "销量": [120, 80, 85, 96, 110]
})

pt = df.pivot_table(
    index="日期", columns="城市", values="销量",
    aggfunc="sum", fill_value=0
)
print("透视表(fill_value=0):")
print(pt)

# --- 执行过程 ---
# 周一对应北京有两行(120 和 80):
#   aggfunc="sum" → 200
# fill_value=0:
#   (日期,城市)组合在原始数据不存在时,填 0 而不是 NaN


> **常见错误**> 1. **错误现象**:报 `ValueError: Index contains duplicate entries`>    **原因**:同一(index,columns)组合有多条记录,但又没传 `aggfunc`。    修复:加上 `aggfunc="sum"` 或 `aggfunc="mean"`。> 2. **错误现象**:透视表里出现大量 NaN>    **原因**:有些组合在原始数据不存在。加 `fill_value=0` 填充即可。

**练习**订单表已有"月份、产品、金额"三列,用 pivot_table 生成"月份×产品"的总金额透视表,空组合显示为 0。> 问自己:> - index / columns / values 分别放哪一列?> - aggfunc 应该选 sum 还是 mean?> - 想让结果多一个合计行,加什么参数?

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "月份": ["1月", "1月", "2月", "2月", "2月"],
    "产品": ["A", "B", "A", "B", "B"],
    "金额": [100, 200, 150, 80, 120]
})

# TODO:月份 x 产品透视表,aggfunc=sum, fill_value=0
pass


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "月份": ["1月", "1月", "2月", "2月", "2月"],
    "产品": ["A", "B", "A", "B", "B"],
    "金额": [100, 200, 150, 80, 120]
})

pt = df.pivot_table(
    index="月份", columns="产品", values="金额",
    aggfunc="sum", fill_value=0
)
print(pt)


#### 数据重塑 melt / stack / unstack —— 宽表 ↔ 长表**痛点**:数据分析师的日常就是"宽↔长"来回切——- 宽表(pivot 后)适合汇报,但需要长表才能画图或建模。- 长表(每行一条观察)适合喂给模型,但不直观。**类比**:- `melt` = 把宽表融化回长表(列名变成一列"变量",值变成一列"值")。- `stack` = 把列级别"堆"到行方向(宽→长)。- `unstack` = 把行级别"掰"到列方向(长→宽,类似 pivot)。**建议**:日常最常见是 `melt` 和 `unstack`。`stack` 涉及多级索引,按需掌握。

In [ ]:
import pandas as pd

# 宽表:列是科目,每行一个学生
wide_df = pd.DataFrame({
    "姓名": ["张三", "李四"],
    "语文": [90, 85],
    "数学": [78, 92]
})

# melt:宽→长,把科目列"融化"成两列(变量名+值)
# id_vars:保留不融化的列(行标识)
# value_vars:要融化的列;不写就融所有非 id 列
long_df = wide_df.melt(
    id_vars="姓名",
    value_vars=["语文", "数学"],
    var_name="科目",
    value_name="分数"
)
print("melt 后(长表):")
print(long_df)

# --- 执行过程 ---
# wide_df.melt(id_vars="姓名", value_vars=["语文","数学"], ...):
#   ① id_vars:"姓名"保留不动
#   ② value_vars:"语文""数学"两列被熔掉
#   ③ 新列"科目"=原来的列名(语文、数学)
#   ④ 新列"分数"=原来格子里的值(90、78 等)
#   ⑤ 结果从 2 行变成 4 行(2 学生 × 2 科目)


**逐行解剖(melt)**- `id_vars`:不需要融化、作为"行标识"的列(保持原值)。- `value_vars`:需要被融化成"变量-值"对的列。不写则融化所有非 id 列。- `var_name` / `value_name`:自定义新列名。默认叫 "variable" / "value"。- `ignore_index`:默认 True,结果索引重新编号。

In [ ]:
import pandas as pd

# 长表:每行一个观察
long_df = pd.DataFrame({
    "姓名": ["张三", "张三", "李四", "李四"],
    "科目": ["语文", "数学", "语文", "数学"],
    "分数": [90, 78, 85, 92]
})

# 多层索引 → unstack 把内层行索引"掰"到列
multi = long_df.set_index(["姓名", "科目"])

# unstack(level=-1):把最内层索引(科目)掰到列
wide_again = multi.unstack(level=-1)
print("unstack 后(宽表):")
print(wide_again)

# 整理:去掉"分数"这层列前缀,让列名只剩"语文""数学"
wide_again.columns = wide_again.columns.droplevel(0)
print("整理列名后:")
print(wide_again)

# --- 执行过程 ---
# set_index(["姓名","科目"]):两层行索引 → MultiIndex
# unstack(level=-1):
#   ① 取最内层索引(科目: 语文/数学)
#   ② 把它们"掰"成列,形成列方向的科目索引
#   ③ 行变单列(姓名),列变成(分数,语文)/(分数,数学)
# droplevel(0):去掉"分数"这层,列名只剩"语文""数学"


> **常见错误**> 1. **错误现象**:melt 后列名变成了"variable" / "value">    **原因**:没传 `var_name` 和 `value_name`。这两个参数是命名关键。> 2. **错误现象**:unstack 后出现 NaN(明明数据里有)>    **原因**:原数据有多行同(姓名,科目),应先 drop_duplicates 或    用 pivot_table 聚合。

**练习**当前宽表为 (姓名、英语、物理、化学),用 melt 融化成长表(姓名、科目、分数),保留"姓名"作为 id 列。> 问自己:> - id_vars 应该是什么? value_vars 又是什么?> - melt 后一共几行?> - 如果想让列名是"科目"和"得分",参数名怎么写?

In [ ]:
import pandas as pd

wide = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "英语": [88, 92, 75],
    "物理": [80, 85, 90],
    "化学": [78, 88, 85]
})

# TODO:melt 融化,id_vars="姓名",var_name="科目",value_name="分数"
pass


In [ ]:
import pandas as pd

wide = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "英语": [88, 92, 75],
    "物理": [80, 85, 90],
    "化学": [78, 88, 85]
})

long = wide.melt(
    id_vars="姓名",
    var_name="科目",
    value_name="分数"
)
print("长表(9 行):")
print(long)


#### 时间序列 resample —— 按时间窗口聚合**痛点**:传感器每分钟一条记录,要看"每天总电量"或"每周平均温度",难道手动 `df[df["时间"].dt.date == ...]` 一天天筛?**类比**:resample 像把一条长线段按"刻度"切成段(10 分钟、1 天、1 月),再对每一段用聚合函数汇总。**解释**:`df.set_index("时间").resample("D").mean()`。- 先把时间列设为索引,索引类型必须是 `DatetimeIndex`。- `"D"`:按天;`"W"`:周;`"M"`:月;`"H"`:小时;`"10min"`:10 分钟。- 调用 `.sum() .mean() .max() .first() .last()` 等聚合。

In [ ]:
import pandas as pd

# 创建含日期+值的 DataFrame
df = pd.DataFrame({
    "日期": pd.date_range("2024-01-01", periods=10, freq="D"),
    "销量": [120, 85, 96, 110, 78, 132, 95, 108, 88, 115]
})
# 把日期列设为索引
df = df.set_index("日期")

# 按周("W")聚合,取每周均值
weekly = df.resample("W").mean()
print("按周平均:")
print(weekly)

# 按月("M")聚合,取每月总销量
monthly = df.resample("M").sum()
print("按月总和:")
print(monthly)

# --- 执行过程 ---
# pd.date_range(起始, periods=10, freq="D"):
#   ① 生成 10 个日期,从 2024-01-01 起,步长 1 天
#   ② 索引即日期
#
# resample("W").mean():
#   ① "W" = 周,按每周一个窗口切分
#   ② 每窗口内求均值,索引变每周日
#
# resample("M").sum():按月切分,求总和


**逐行解剖(resample)**- 必须先 `set_index("时间列")`,索引必须是 `DatetimeIndex`。- 频率别名:`D`天、`W`周、`M`月、`H`小时、`T/min`分钟、`S`秒。- `.resample("2D").sum()`:每 2 天聚合一次。- `label="left"` / `label="right"`:决定窗口标签落在左/右边界。

In [ ]:
import pandas as pd

# 把字符串日期转成真正的 Timestamps,然后用 resample
df = pd.DataFrame({
    "日期": ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-05"],
    "温度": [6.5, 7.0, 5.8, 4.2]
})

# pd.to_datetime:把字符串转成 Timestamp
df["日期"] = pd.to_datetime(df["日期"])
df = df.set_index("日期")

# resample("D"):按日历日切窗口,缺失日(01-04)也出现(值 NaN)
daily = df.resample("D").mean()
print("按天重采样(01-04 出现 NaN):")
print(daily)

# ffill 前向填充:用前一天有效值填补 NaN 日
filled = daily.ffill()
print("前向填充后:")
print(filled)

# --- 执行过程 ---
# pd.to_datetime:字符串 → Timestamp,后续 resample 才能识别
# resample("D").mean():
#   ① 按日历日切窗口,缺失日也出现(值为 NaN)
#   ② .ffill() 用前一天的值填充,补上 01-04 的缺口


> **常见错误**> 1. **错误现象**:`Only valid with DatetimeIndex...`>    **原因**:索引还不是 DatetimeIndex。要先 pd.to_datetime + set_index。> 2. **错误现象**:resample 后整列都是 NaN>    **原因**:字符串日期没转成 datetime,没有任何窗口能和数据对齐。

**练习**股票价格表有"日期"和"收盘价"两列(共 5 天数据,跳跃分布)。请 resample 到**每天**,均值,缺失日的前一天价格前向填充。> 问自己:> - pd.to_datetime 应该用在哪一列?> - resample 的频率字符串应该传什么?> - 除了 ffill,还可以用哪些填充策略?

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "日期": ["2024-01-01", "2024-01-02", "2024-01-04",
             "2024-01-05", "2024-01-07"],
    "收盘价": [100, 102, 101, 105, 110]
})

# TODO:转 datetime → set_index → resample(D) → 用 ffill 填补缺失日
pass


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "日期": ["2024-01-01", "2024-01-02", "2024-01-04",
             "2024-01-05", "2024-01-07"],
    "收盘价": [100, 102, 101, 105, 110]
})

df["日期"] = pd.to_datetime(df["日期"])
df = df.set_index("日期")
daily = df.resample("D").mean()
print("按天(missing 日呈 NaN):")
print(daily)

filled = daily.ffill()
print("ffill 后:")
print(filled)


#### 文本处理 str 访问器 —— 给整列字符串批量操作**痛点**:名字列全是"姓 名"格式,想拆成"姓"和"名"两列。难道写 for 循环每条处理?万一有百万行就慢了。**类比**:Pandas 的 `.str` 是一把"批处理瑞士军刀",内部向量化,快。常用操作:- `.str.lower() / upper()`   —— 大小写转换- `.str.strip()`             —— 删首尾空格- `.str.contains("模式")`    —— 模糊匹配返回布尔 Series- `.str.split("分隔符", expand=True)` —— 拆成多列- `.str.replace("旧", "新")`  —— 替换**解释**:`df["列名"].str.方法(参数)`。对整列每个元素调用同一字符串方法。

In [ ]:
import pandas as pd

# 数据:姓名列有首尾空格,城市列大小写混乱
df = pd.DataFrame({
    "姓名": [" 张 三 ", " 李四", "王 五 ", "赵  六"],
    "城市": ["beijing", "Shanghai", "GUANGZHOU", "shenzhen"]
})

# strip:去首尾空格;replace:把冗余的空格全删掉
df["姓名"] = df["姓名"].str.strip().str.replace(" ", "")

# upper/lower:大小写统一
df["城市"] = df["城市"].str.upper()

print(df)

# --- 执行过程 ---
# .str.strip() → .str.replace(" ", ""):
#   ① strip 先去首尾空格:" 张 三 " → "张 三"
#   ② replace(" ", "") 把中间空格全删:"张 三" → "张三"
#
# .str.upper():大小写统一 → "beijing" → "BEIJING"


**逐行解剖**- `.str.upper() .lower() .title()`:大小写转换,统一格式后再合并不漏。- `.str.strip()`:删首尾空格/换行符/制表符,真实脏数据里经常出现。- `.str.contains("A")`:返回布尔 Series,可当过滤条件(类似 SQL LIKE)。- `expand=True`:split 结果展开成多列;不写默认返回 list。

In [ ]:
import pandas as pd

emails = pd.Series([
    "zhangsan@gmail.com",
    "lisi@qq.com",
    "wangwu@163.com"
])

# split("@", expand=True) 拆成两列
parts = emails.str.split("@", expand=True)
print("split expand=True:")
print(parts)
print("---")

# contains 模糊匹配,返回布尔 Series
is_qq = emails.str.contains("qq")
print("哪些是 QQ 邮箱:")
print(is_qq)

# --- 执行过程 ---
# .str.split("@", expand=True):
#   ① 以 "@" 为分隔符拆每个元素
#   ② expand=True:每段各占一列 → 0=用户名, 1=域名
#   ③ 不写 expand 则返回 list,仍是单列
#
# .str.contains("qq"):
#   ① 对每个元素判断是否包含子串 "qq"
#   ② 返回布尔 Series,可当索引: emails[is_qq] 过滤


> **常见错误**> 1. **错误现象**:`AttributeError: Can only use .str accessor with string values`>    **原因**:该列含有 NaN(float 类型),直接 .str 会报错。    解决:先 `.fillna("")` 把 NaN 替成空字符串。> 2. **错误现象**:replace 后原数据没变化>    **原因**:`.str.replace` 返回新 Series,需赋值回去,如    `df["列"] = df["列"].str.replace("a", "b")`。

**练习**邮箱列表列有 "name@example.com" 格式。用 `.str.split("@", expand=True)`拆出"用户名"和"域名"两列,存入新列。> 问自己:> - expand=True 不写会得到什么?> - 如果邮箱有大写,transform 统一小写要怎么做?> - 如果列里有 NaN,如何处理?

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "邮箱": ["zhangsan@gmail.com", "lisi@qq.com", "wangwu@163.com"]
})

# TODO:split("@", expand=True) 拆成两列,存入 df
pass


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "邮箱": ["zhangsan@gmail.com", "lisi@qq.com", "wangwu@163.com"]
})

parts = df["邮箱"].str.split("@", expand=True)
df["用户名"] = parts[0]
df["域名"] = parts[1]
print(df)


#### 分类数据 Categorical —— 省内存,保证顺序**痛点**:一列性别只有 3 种取值(男/女/未知),但每行都存一个字符串"男"。如果 1000 万行,内存爆炸。而且 sort 时"男","女"按字母序排,不是按"等级"排(比如 低<中<高)。**类比**:Categorical 像"定好三个抽屉(男/女/未知)"——- 内部抽屉编号存数字(0/1/2),省内存。- 显示时自动还原为字符串"男/女/未知"。- 可以定义顺序,让 sort 按业务等级排。**解释**:- `df["列"].astype("category")`:最简单创建。- `pd.CategoricalDtype(categories=[...], ordered=True)`:定义有序分类。- 编码:`df["列"].cat.codes`。类别:`df["列"].cat.categories`。

In [ ]:
import pandas as pd

# 重复值很多的字符串列(如学历等级)
df = pd.DataFrame({
    "学生": ["张三", "李四", "王五", "赵六", "孙七", "钱八"],
    "学历": ["本科", "硕士", "本科", "博士", "本科", "硕士"]
})

# 转成 category:内部数字代号,省内存
df["学历"] = df["学历"].astype("category")
print("dtype:", df["学历"].dtype)
print("内部编码 .cat.codes:")
print(df["学历"].cat.codes)
print("类别列表 .cat.categories:")
print(df["学历"].cat.categories.tolist())

# --- 执行过程 ---
# .astype("category"):
#   ① 识别所有唯一值:本科、硕士、博士
#   ② 内部用 0/1/2 代号存储,大幅省内存
#   ③ 对外显示不变
#
# .cat.codes:返回内部数字编码(int8)
# .cat.categories:所有类别,按出现顺序


**逐行解剖**- `.cat.codes`:内部整数码,可喂给只接受数字的机器学习模型。- `.cat.categories`:类别列表,顺序决定 unordered 时的默认顺序。- `ordered=True`:定义顺序后,`.sort_values()` 按该顺序排,不再按字母序。

In [ ]:
import pandas as pd
from pandas.api.types import CategoricalDtype

# 有序分类:定义 "低" < "中" < "高"
cat_type = CategoricalDtype(
    categories=["低", "中", "高"],
    ordered=True
)

df = pd.DataFrame({
    "产品": ["P1", "P2", "P3", "P4"],
    "等级": ["高", "低", "中", "低"]
})

df["等级"] = df["等级"].astype(cat_type)

# sort_values:按定义好的顺序排(不再是字母序)
sorted_df = df.sort_values("等级")
print("自定义顺序排序结果:")
print(sorted_df)

# --- 执行过程 ---
# CategoricalDtype(categories=["低","中","高"], ordered=True):
#   ① 显式定义类别顺序:低<中<高
#   ② ordered=True 表示有序
#
# df["等级"].astype(cat_type):
#   ① 按指定类别+顺序转换
#   ② 不在 categories 的值会变成 NaN(要注意!)
#
# sort_values("等级"):按定义好的顺序排,不再是字母序


> **常见错误**> 1. **错误现象**:赋值时报 `ValueError: fill value must be in categories`>    **原因**:新值不在 categories 里。应先    `df["列"].cat.add_categories(["新值"])` 添加类别。> 2. **错误现象**:sort_values 仍按字母排>    **原因**:没有 `ordered=True`,类别默认是无序的。

**练习**评分列有 A/B/C/D 四个等级,创建有序 CategoricalDtype(A<B<C<D),然后用 sort_values 按等级排序。> 问自己:> - categories 列表顺序应该怎么写?> - ordered 应该传什么?> - 如果有个 E 等级不在 categories 里,会怎样?

In [ ]:
import pandas as pd
from pandas.api.types import CategoricalDtype

df = pd.DataFrame({
    "课程": ["数学", "语文", "英语", "物理"],
    "等级": ["C", "A", "D", "B"]
})

# TODO:创建有序 CategoricalDtype(A<B<C<D),astype + sort_values
pass


In [ ]:
import pandas as pd
from pandas.api.types import CategoricalDtype

df = pd.DataFrame({
    "课程": ["数学", "语文", "英语", "物理"],
    "等级": ["C", "A", "D", "B"]
})

cat_type = CategoricalDtype(
    categories=["A", "B", "C", "D"],
    ordered=True
)
df["等级"] = df["等级"].astype(cat_type)
sorted_df = df.sort_values("等级")
print("按等级排序:")
print(sorted_df)


#### 数据导入导出 —— read_csv / read_excel / to_csv / to_excel**痛点**:数据分析从读文件开始。CSV 是最常见格式,但编码、日期、缺失值处理不当会导致读错或报错。处理完的数据又要保存回去。**类比**:read_csv/excel 是"开门",to_csv/excel 是"关门"——关键是调对参数,不然开得像啃硬骨头,关得像丢东西。**解释**:- `df = pd.read_csv("路径.csv")`:读 UTF-8 编码 CSV 文件。- `df.to_csv("out.csv", index=False, encoding="utf-8-sig")`:  写入,`index=False` 不保存行索引。- `pd.read_excel("路径.xlsx")`:读 Excel 文件。- `df.to_excel("out.xlsx", index=False, sheet_name="Sheet1")`:写入 Excel。

In [ ]:
import pandas as pd

# 第一步:构造示例数据 DataFrame
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "年龄": [28, 35, 42],
    "城市": ["北京", "上海", "深圳"]
})

# 写 CSV,index=False 避免把行索引写入
df.to_csv("/tmp/pd_sample.csv", index=False, encoding="utf-8")

# 读回 CSV
df_read = pd.read_csv("/tmp/pd_sample.csv")
print("读回 DataFrame:")
print(df_read)

# --- 执行过程 ---
# to_csv("/tmp/pd_sample.csv", index=False):
#   ① 把 DataFrame 写到磁盘
#   ② index=False:必写,否则多一列行索引
#
# read_csv("/tmp/pd_sample.csv"):
#   ① 第一行自动识别为列名
#   ② 当前编码 utf-8 能正确读取中文


**逐行解剖**- `pd.read_csv()` 常用参数:  `sep`(分隔符,默认 `,`)、`header`(哪行当列名)、`encoding`  (编码,常见 utf-8、gbk)、`parse_dates`(指定日期列)、  `na_values`(标记缺失值,如 "N/A" "-")。- `df.to_csv(path, index=False, encoding="utf-8-sig")`:  - `index=False` 必写,否则第一列是"Unnamed: 0"。  - `encoding="utf-8-sig"`:带 BOM 的 UTF-8,Excel 打开不乱码。- `pd.read_excel("xlsx", sheet_name="名称")`:按工作表名读取。- `df.to_excel()`:同样 index=False 必写,`sheet_name` 命名。

In [ ]:
import pandas as pd

# 构造含缺失值标记的数据
df = pd.DataFrame({
    "产品": ["A", "B", "C", "D"],
    "销量": [100, None, -1, 50]
})

# 写 CSV(缺失值 NaN 会变成空单元格)
df.to_csv("/tmp/pd_na.csv", index=False)

# 读取时指定"-1"也算缺失值
df_na = pd.read_csv("/tmp/pd_na.csv", na_values=["-1", "NA"])
print("指定缺失值标记后:")
print(df_na)

# Excel 读取示例(注释,避免无 openpyxl 时报错):
# df_xlsx = pd.read_excel("data.xlsx", sheet_name="Sheet1")

# --- 执行过程 ---
# to_csv(path, index=False):写到 disk,NaN 变成空单元格
#
# read_csv(path, na_values=["-1","NA"]):
#   ① 读取时把 "-1" 和 "NA" 视作缺失值,统一成 NaN
#   ② 后续 isnull().sum() 可以统计缺失情况
#
# read_excel(path, sheet_name="Sheet1"):
#   ① 读取指定工作表的全部数据


> **常见错误**> 1. **错误现象**:读 CSV 后多了一列 `Unnamed: 0`,看起来像行索引>    **原因**:写入时没设 `index=False`。修复:`df.to_csv(path, index=False)`。> 2. **错误现象**:`UnicodeDecodeError` 读 CSV 报错>    **原因**:文件是 GBK 编码,默认按 UTF-8 解析。加 `encoding="gbk"`。> 3. **错误现象**:Excel 日期读出来是数字 45000 几>    **原因**:Excel 内部日期是序列号。加 `parse_dates=["日期列"]`。

**练习**把下面 CSV 字符串(含缺失值标记"暂无")用 read_csv 读入,指定"暂无"为缺失值,然后再 to_csv 写出到 /tmp/out.csv。> 问自己:> - na_values 应该传什么?> - index 参数应该传什么?> - 写出后第一列是"姓名"还是"Unnamed: 0"?

In [ ]:
import pandas as pd
from io import StringIO

# 模拟 CSV 文本(缺失格用"暂无"标记)
csv_text = """姓名,年龄
张三,28
李四,暂无
王五,35"""

# TODO:pd.read_csv(StringIO(csv_text), na_values=["暂无"])
# TODO:然后 df.to_csv("/tmp/out.csv", index=False)
pass


In [ ]:
import pandas as pd
from io import StringIO

csv_text = """姓名,年龄
张三,28
李四,暂无
王五,35"""

# na_values:指定"暂无"为缺失值
# index=False:不写出行索引
df = pd.read_csv(
    StringIO(csv_text),
    na_values=["暂无"]
)
print(df)

df.to_csv("/tmp/out.csv", index=False)
print("写出完成")


**今日小结**| 知识点 | 函数 | 解决的痛点 ||---|---|---|| 分组聚合 | `.groupby().agg()` | 按类别一次性多指标汇总 || 表合并 | `.merge(on, how)` | 像 SQL JOIN 按 key 拼表 || 多表拼接 | `.concat(axis)` | 纵向堆叠或横向接表 || 透视表 | `.pivot_table()` | 长→宽矩阵交叉统计 || 数据重塑 | `.melt() / .unstack()` | 宽 ↔ 长互转 || 时间序列 | `.resample()` | 按时间窗口聚合 || 文本处理 | `.str.*` | 列级批处理 split/replace/contains || 分类数据 | `CategoricalDtype` | 省内存、定义类别顺序 || 导入导出 | `read_csv/to_csv`、`read_excel/to_excel` | 读文件、写文件 |**更多练习**- 当堂练:`course/lesson17/in_class/practice01.py` ~ `practice08.py`- 课后作业:`course/lesson17/homework/task01.py` ~ `task03.py`